# YOLOv8 Experiments: GOST Stamp Detection

**Цель:** Обучить YOLOv8 для детекции штампов на строительных чертежах.

**v2 (исторический):** 500 synthetic (80/20 train/val) + 49 real (test, 4 donors excluded)
**v3 (чистый baseline):** 500 synthetic (all train) + 10 real stratified val (val_honest) + 35 real test (4 donors + 10 val excluded)

**Метрики:** IoU, Precision, Recall, F1 на 35 реальных non-donor, non-val изображениях

**Подход:** Single training run, yolov8n, 50 epochs, GPU T4 (Colab)


# 1. Setup

⚠️ **Запустить только один раз!** Клонирует репозиторий (sparse checkout) и устанавливает зависимости.


In [ ]:
import sys
import random
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    %cd /content
    !rm -rf aie-group-2-sapar
    !git init aie-group-2-sapar
    %cd aie-group-2-sapar
    !git sparse-checkout set project
    !git remote add origin https://github.com/Sapar-hub/aie-group-2-sapar.git
    !git pull origin main
    %cd project
    !pip install -q ultralytics opencv-python-headless pyyaml
    PROJECT_DIR = Path.cwd()
    sys.path.insert(0, str(PROJECT_DIR / "src"))
else:
    PROJECT_DIR = Path.cwd().parent
    sys.path.insert(0, str(PROJECT_DIR / "src"))


In [ ]:
# Load pre-generated training data from Google Drive (Colab only)
if IN_COLAB and not (Path('data') / 'images' / 'train_v2').exists():
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil
    DRIVE_DATA = Path('/content/drive/MyDrive/aie-group-2-data')
    if (DRIVE_DATA / 'images' / 'train_v2').exists():
        for subdir in ['images', 'labels']:
            shutil.copytree(str(DRIVE_DATA / subdir), str(Path('data') / subdir), dirs_exist_ok=True)
        print("Training data loaded from Google Drive")
    else:
        print("=" * 60)
        print("No pre-generated data found on Drive!")
        print("Run notebooks/exp02_synthetic_data.ipynb first to generate it.")
        print("=" * 60)


## 2. Imports & Data Setup

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import yaml
from ultralytics import YOLO

from evaluation.metrics import DetectionResult, bbox_iou, yolo_to_pixel, compute_metrics, print_metrics
from data.loader import load_image_and_labels

RANDOM_STATE = 42

with open(PROJECT_DIR / "configs" / "config.yaml") as f:
    CFG = yaml.safe_load(f)

DATA_DIR = PROJECT_DIR / "data"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)
(ARTIFACTS_DIR / "models").mkdir(exist_ok=True)
(ARTIFACTS_DIR / "metrics").mkdir(exist_ok=True)
(ARTIFACTS_DIR / "figures").mkdir(exist_ok=True)

IMAGE_TEST_DIR = DATA_DIR / "images" / "test"
LABEL_TEST_DIR = DATA_DIR / "labels" / "test"

print(f"Working dir: {PROJECT_DIR}")
print(f"Test images: {len(list(IMAGE_TEST_DIR.glob('*.png'))) + len(list(IMAGE_TEST_DIR.glob('*.jpg')))}")
print(f"Test labels: {len(list(LABEL_TEST_DIR.glob('*.txt')))}")

Working dir: /home/saparch/playground/aie-group-2-sapar/project
Test images: 49
Test labels: 49


## 3. Data, Donors & Split (v3 Clean Baseline)

**Donors (4):** stamp sources for copy-paste synthesis, excluded from eval.
**Val (10):** stratified-selected real images for YOLO validation during training.
**Test (35):** remaining non-donor, non-val images for final evaluation.

All 49 real images loaded for inference; 14 excluded from metrics (4 donors + 10 val) → **eval on 35 test images**.


In [3]:
# Donors and val images are excluded from eval metrics
# Stratified val selection runs in exp02_synthetic_data.ipynb
DONORS = {"test_09.jpg", "test_14.png", "test_34.jpg", "test_49.jpg"}
print(f"Donors (stamp sources, excluded from eval): {sorted(DONORS)}")

# Val images are selected by select_donors() with exclude=DONORS, n_donors=10, seed=42
# They are copied to data/images/val_honest/ by exp02
VAL_HONEST_DIR = DATA_DIR / "images" / "val_honest"
VAL_IMAGES = set(p.name for p in sorted(VAL_HONEST_DIR.glob("*.png")) + sorted(VAL_HONEST_DIR.glob("*.jpg")))
print(f"Val images (real, stratified, excluded from test eval): {sorted(VAL_IMAGES)}")
print(f"Test set: 49 - {len(DONORS)} - {len(VAL_IMAGES)} = {49 - len(DONORS) - len(VAL_IMAGES)} images")


Donors (stamp sources, excluded from eval): ['test_09.jpg', 'test_14.png', 'test_34.jpg', 'test_49.jpg']
Non-donors (clean eval set): 49 - 4 = 45


In [4]:
# All 49 real images loaded for inference; 4 donors + 10 val excluded from metrics
import glob
all_images = sorted(IMAGE_TEST_DIR.glob("*.png")) + sorted(IMAGE_TEST_DIR.glob("*.jpg"))
test_images = all_images  # no holdout split, DONORS + VAL_IMAGES filter in metrics

EXCLUDE = DONORS | VAL_IMAGES
print(f"Test images: {len(test_images)} (eval on {len(test_images) - len(EXCLUDE)} non-donor, non-val after filter)")


Test images: 49 (eval on 45 non-donors after filter)


## 4. Training (v3)

Запускаем YOLOv8n на данных из `gost_stamp_v3.yaml`.
Параметры: `rect=True` (сохраняет пропорции A4 vs A1), `mosaic=0.0` (не режет мелкие штампы), `seed=42` (воспроизводимость).

**v3 отличие:** val теперь указывает на 10 реальных изображений (val_honest), а не на синтетику.


In [ ]:
from models.train_yolo import train_yolo
import time
start = time.time()

best_pt = train_yolo(
    config_path=PROJECT_DIR / "configs" / "config.yaml",
    yaml_name="gost_stamp_v3.yaml",
    project=str(ARTIFACTS_DIR / "yolo"),
    name="exp01",
)

elapsed = time.time() - start
print(f"\nTraining time: {elapsed/60:.1f} minutes")

## 5. Evaluation — Score Threshold Sweep + Greedy Matching

Загружаем лучшие веса, оцениваем на 35 test non-donor (4 донора + 10 val исключены из метрик).
Sweep по `conf ∈ [0.05, 0.1, 0.2, 0.3]`, выбор по F1.
Greedy matching: из нескольких предсказаний выбираем с лучшим IoU к GT.
Fallback: если greedy не нашёл — берём первый prediction (highest confidence).


In [ ]:
from evaluation.evaluate_yolo import evaluate_yolo

model = YOLO(str(best_pt))
print(f"Loaded weights from {best_pt}")

best_metrics, results_list = evaluate_yolo(
    model, IMAGE_TEST_DIR, LABEL_TEST_DIR, DONORS
)
print_metrics(best_metrics, prefix="\nYOLO ")
metrics = best_metrics

## 6. IoU Threshold Analysis

In [7]:
print("IoU @ different thresholds:")
for thresh in [0.3, 0.5, 0.75]:
    m = compute_metrics(results_list, iou_threshold=thresh)
    print(f"  IoU >= {thresh}: {m.get('iou_at_threshold', 0)*100:.1f}%")

ious = [r.iou for r in results_list]
print(f"\nIoU stats: mean={np.mean(ious):.3f}, std={np.std(ious):.3f}, median={np.median(ious):.3f}")
print(f"Detection rate: {sum(1 for r in results_list if r.found)}/{len(results_list)}")

IoU @ different thresholds:
  IoU >= 0.3: 84.4%
  IoU >= 0.5: 84.4%
  IoU >= 0.75: 84.4%

IoU stats: mean=0.765, std=0.329, median=0.906
Detection rate: 38/45


## 7. Visualization

In [8]:
import cv2

sorted_results = sorted(results_list, key=lambda r: r.iou)
worst = sorted_results[0]
best = sorted_results[-1]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, result, title_prefix in zip(axes, [worst, best], ["Worst", "Best"]):
    img_path = [p for p in test_images if p.name == result.image_name][0]
    img, _ = load_image_and_labels(img_path, LABEL_TEST_DIR)
    vis = img.copy()
    
    if result.gt_bbox:
        x, y, bw, bh = result.gt_bbox
        cv2.rectangle(vis, (x, y), (x+bw, y+bh), (0, 255, 0), 3)
    if result.pred_bbox:
        x, y, bw, bh = result.pred_bbox
        cv2.rectangle(vis, (x, y), (x+bw, y+bh), (0, 0, 255), 2)
    
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{title_prefix} IoU={result.iou:.3f}")
    ax.axis("off")

plt.suptitle("Green=GT, Red=Pred (YOLO)")
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "figures" / "yolo_best_worst.png", dpi=150)
plt.show()

<Figure size 1600x800 with 2 Axes>

## 8. Conclusions (v3 Clean Baseline)

Threshold sweep + greedy matching на 35 test non-donor images (4 donors + 10 val excluded).
Лучший `conf` выбран по F1.

**v3 отличие:** val split — 10 реальных изображений (val_honest, стратифицированный отбор).
Тест — 35 реальных изображений, никогда не использовавшихся в обучении.


In [ ]:
summary = {
    "model": "YOLOv8n",
    "data": "500 synthetic (all train) + 49 real (eval on 35 test non-donor, non-val)",
    "epochs": 50,
    "imgsz": 640,
    "rect": True,
    "mosaic": 0.0,
    "train_time_min": round(elapsed/60, 1),
    "iou_mean": round(metrics['iou_mean'], 3),
    "iou_std": round(metrics['iou_std'], 3),
    "precision": round(metrics['precision'], 3),
    "recall": round(metrics['recall'], 3),
    "f1": round(metrics['f1'], 3),
    "detection_rate": round(metrics['detection_rate'], 3),
}

import json
with open(ARTIFACTS_DIR / "metrics" / "yolo_v3_results.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Summary saved to artifacts/metrics/yolo_v3_results.json")
print(json.dumps(summary, indent=2))